In [ ]:
pip install hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 23.4 MB/s eta 0:00:00


In [ ]:
import os
print(os.getcwd())

In [ ]:
!git clone https://github.com/Jakobovski/free-spoken-digit-dataset.git

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt

# Choose one audio file
file_path = DATA_DIR / "0_george_0.wav"
# Load audio
y, sr = librosa.load(file_path, sr=None, mono=True)
# Remove silence
y, _ = librosa.effects.trim(y, top_db=20)
# Normalize
y = y / np.max(np.abs(y))
# Apply Hanning window
N = len(y)
x = y * np.hanning(N)
# Compute FFT
X = np.fft.rfft(x)
# Magnitude spectrum
mag = np.abs(X)
# Frequency axis
freqs = np.fft.rfftfreq(N, d=1/sr)
# Plot spectrum
plt.figure(figsize=(10, 4))
plt.plot(freqs, mag)
plt.title("Spectrum: 0_george_0.wav")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.grid(True)
plt.show()

def extract_dft_features(file_path):
    y, sr = librosa.load(file_path, sr=None, mono=True)
    y, _ = librosa.effects.trim(y, top_db=20)
     # Remove silence from beginning and end
    y = y / np.max(np.abs(y))
    # Normalize signal amplitude to range [-1, 1]


    N = len(y)
    x = y * np.hanning(N) # Apply Hanning window to reduce spectral leakage


    X = np.fft.rfft(x)
    mag = np.abs(X) # Compute magnitude spectrum (strength of each frequency)
    freqs = np.fft.rfftfreq(N, d=1/sr)  # Get frequency values corresponding to FFT bins

    power = mag ** 2
    power_sum = np.sum(power) + 1e-12

    # Feature 1
    centroid = np.sum(freqs * power) / power_sum # Spectral centroid: weighted average of frequencies (center of energy)
    # Feature 2
    bandwidth = np.sqrt(
        np.sum(((freqs - centroid) ** 2) * power) / power_sum
    ) # Spectral bandwidth: spread of frequencies around the centroid

    cumulative_energy = np.cumsum(power)
    rolloff_threshold = 0.85 * cumulative_energy[-1]
    rolloff_idx = np.searchsorted(cumulative_energy, rolloff_threshold)
    rolloff = freqs[min(rolloff_idx, len(freqs) - 1)] # Frequency where 85% of energy is contained #Feature 3

    mag_no_dc = mag.copy() # Copy magnitude spectrum to avoid modifying original
    mag_no_dc[0] = 0 # Remove the DC component
    dominant_idx = np.argmax(mag_no_dc) # Find index of the strongest frequency (highest magnitude)
    dominant_freq = freqs[dominant_idx] #Feature 4

    p = power / power_sum
    #Feature 5
    entropy = -np.sum(p * np.log2(p + 1e-12)) # Compute spectral entropy (measure of randomness in frequency distribution)

    #Feature 6
    def band_energy_ratio(f_low, f_high):
        idx = np.where((freqs >= f_low) & (freqs < f_high))[0]
        if len(idx) == 0:
            return 0.0
        return np.sum(power[idx]) / power_sum # Measure energy distribution across frequency bands

    band_0_500 = band_energy_ratio(0, 500)
    band_500_1000 = band_energy_ratio(500, 1000)
    band_1000_2000 = band_energy_ratio(1000, 2000)
    band_2000_4000 = band_energy_ratio(2000, 4000)

    #Feature 6
    flatness = np.exp(np.mean(np.log(mag + 1e-12))) / (np.mean(mag) + 1e-12) # Spectral flatness: indicates whether the signal is noise-like or tone-like

    mean_freq = centroid
    #Feature 7
    skewness = np.sum(((freqs - mean_freq)**3) * power) / (power_sum * (bandwidth**3 + 1e-12)) # Skewness: asymmetry of frequency distribution
    #Feature 8
    kurtosis = np.sum(((freqs - mean_freq)**4) * power) / (power_sum * (bandwidth**4 + 1e-12)) # Kurtosis: sharpness of spectral peak
    #Feature 9
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))




    return [
        centroid,
        bandwidth,
        rolloff,
        dominant_freq,
        entropy,
        band_0_500,
        band_500_1000,
        band_1000_2000,
        band_2000_4000,
        flatness,
        skewness,
        kurtosis,
        zcr
    ]

# Build dataset: extract features and labels from all audio files
X = []
y = []

import os

for file_name in os.listdir(DATA_DIR):
    if file_name.endswith(".wav"):
        file_path = DATA_DIR / file_name
        digit = int(file_name.split("_")[0])
        features = extract_dft_features(file_path)

        X.append(features)
        y.append(digit)


In [ ]:
# Build dataset: extract features and labels from all audio files
X = []
y = []

import os

for file_name in os.listdir(DATA_DIR):
    if file_name.endswith(".wav"):
        file_path = DATA_DIR / file_name
        digit = int(file_name.split("_")[0])
        features = extract_dft_features(file_path)

        X.append(features)
        y.append(digit)

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)


# Split dataset into training and testing sets (90% train, 10% test)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# Scale features to reduce effect of outliers
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(X_train.shape)
print(X_test.shape)


# Train Random Forest model and evaluate accuracy
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    min_samples_leaf=1,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import librosa


# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



# Feature Importance
feature_names = [
    "spectral_centroid",
    "spectral_bandwidth",
    "spectral_rolloff_85",
    "dominant_freq",
    "spectral_entropy",
    "spectral_flatness",
    "band_0_500",
    "band_500_1000",
    "band_1000_2000",
    "band_2000_4000",
    "skewness",
    "kurtosis",
    "zcr"
]

importances = model.feature_importances_

df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

df = df.sort_values(by="importance", ascending=False)

print("\nFeature Importance:\n")
print(df)


# Plot Feature Importance
plt.figure(figsize=(8, 5))
plt.barh(df["feature"], df["importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.show()


# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


# Windowing Effect: Rectangular vs Hanning
file_path = DATA_DIR / "0_george_0.wav"

y, sr = librosa.load(file_path, sr=None, mono=True)
y, _ = librosa.effects.trim(y, top_db=20)

max_abs = np.max(np.abs(y))
if max_abs > 0:
    y = y / max_abs

# Rectangular window = no window applied
X_rect = np.fft.rfft(y)
mag_rect = np.abs(X_rect)
freqs = np.fft.rfftfreq(len(y), d=1/sr)

# Hanning window
hanning_window = np.hanning(len(y))
y_hanning = y * hanning_window

X_hanning = np.fft.rfft(y_hanning)
mag_hanning = np.abs(X_hanning)

plt.figure(figsize=(12, 5))
plt.plot(freqs, mag_rect, label="Rectangular Window", alpha=0.7)
plt.plot(freqs, mag_hanning, label="Hanning Window", alpha=0.7)

plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("Windowing Effect on Spectrum")
plt.legend()
plt.grid(True)
plt.show()


# Window Length Effect - Separate Plots
window_lengths = [256, 512, 1024, len(y)]


max_mag = 0
results = []

for L in window_lengths:
    if L > len(y):
        L = len(y)

    x = y[:L]

    window = np.hanning(len(x))
    x_windowed = x * window

    X_fft = np.fft.rfft(x_windowed)
    mag = np.abs(X_fft)
    freqs = np.fft.rfftfreq(len(x), d=1/sr)

    results.append((L, freqs, mag))
    max_mag = max(max_mag, np.max(mag))
# The following visualization code was created with assistance from ChatGPT
for L, freqs, mag in results:
    plt.figure(figsize=(10, 4))
    plt.plot(freqs, mag)

    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.title(f"Effect of Window Length on Spectrum (L = {L})")

    plt.xlim(0, 4000)
    plt.ylim(0, max_mag)

    plt.grid(True)
    plt.show()

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Function: Extract DFT features using specific FFT size
def extract_dft_features_N(file_path, N_FFT):
    y, sr = librosa.load(file_path, sr=None, mono=True)

    y, _ = librosa.effects.trim(y, top_db=20)

    max_abs = np.max(np.abs(y))
    if max_abs > 0:
        y = y / max_abs

    x = y.copy()

    # Zero-padding only if N_FFT is larger than signal length
    if N_FFT > len(x):
        x = np.pad(x, (0, N_FFT - len(x)))

    # Apply Hanning window
    x = x * np.hanning(len(x))

    # Compute FFT using N_FFT if padded, otherwise full signal length
    fft_len = len(x)

    X = np.fft.rfft(x, n=fft_len)
    mag = np.abs(X)
    freqs = np.fft.rfftfreq(fft_len, d=1/sr)

    power = mag ** 2
    power_sum = np.sum(power) + 1e-12

    centroid = np.sum(freqs * power) / power_sum

    bandwidth = np.sqrt(
        np.sum(((freqs - centroid) ** 2) * power) / power_sum
    )

    cumulative_energy = np.cumsum(power)
    rolloff_threshold = 0.85 * cumulative_energy[-1]
    rolloff_idx = np.searchsorted(cumulative_energy, rolloff_threshold)
    rolloff = freqs[min(rolloff_idx, len(freqs) - 1)]

    mag_no_dc = mag.copy()
    mag_no_dc[0] = 0
    dominant_idx = np.argmax(mag_no_dc)
    dominant_freq = freqs[dominant_idx]

    p = power / power_sum
    entropy = -np.sum(p * np.log2(p + 1e-12))

    flatness = np.exp(np.mean(np.log(mag + 1e-12))) / (np.mean(mag) + 1e-12)

    def band_energy_ratio(f_low, f_high):
        idx = np.where((freqs >= f_low) & (freqs < f_high))[0]
        if len(idx) == 0:
            return 0.0
        return np.sum(power[idx]) / power_sum

    band_0_500 = band_energy_ratio(0, 500)
    band_500_1000 = band_energy_ratio(500, 1000)
    band_1000_2000 = band_energy_ratio(1000, 2000)
    band_2000_4000 = band_energy_ratio(2000, 4000)

    mean_freq = centroid

    skewness = np.sum(((freqs - mean_freq) ** 3) * power) / (
        power_sum * (bandwidth ** 3 + 1e-12)
    )

    kurtosis = np.sum(((freqs - mean_freq) ** 4) * power) / (
        power_sum * (bandwidth ** 4 + 1e-12)
    )
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))

    return [
        centroid,
        bandwidth,
        rolloff,
        dominant_freq,
        entropy,
        flatness,
        band_0_500,
        band_500_1000,
        band_1000_2000,
        band_2000_4000,
        skewness,
        kurtosis,
        zcr
    ]


# Experiment: Test different FFT sizes
for N in [256, 512, 1024, 2048, 4096]:

    X_exp = []
    y_exp = []

    for file_name in os.listdir(DATA_DIR):
        if file_name.endswith(".wav"):
            file_path = DATA_DIR / file_name
            digit = int(file_name.split("_")[0])

            features = extract_dft_features_N(file_path, N)

            X_exp.append(features)
            y_exp.append(digit)

    X_exp = np.array(X_exp)
    y_exp = np.array(y_exp)

    X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
        X_exp,
        y_exp,
        test_size=0.1,
        random_state=42,
        stratify=y_exp
    )

    model = RandomForestClassifier(
        n_estimators=400,
        max_depth=20,
        min_samples_leaf=1,
        random_state=42
    )

    model.fit(X_train_exp, y_train_exp)

    y_pred_exp = model.predict(X_test_exp)
    acc = accuracy_score(y_test_exp, y_pred_exp)

    print(f"N_FFT = {N}  Accuracy = {acc:.6f}")

# The following visualization code was created with assistance from ChatGPT
# Frequency resolution
file_path = DATA_DIR / "0_george_0.wav"

y, sr = librosa.load(file_path, sr=None, mono=True)
y, _ = librosa.effects.trim(y, top_db=20)

max_abs = np.max(np.abs(y))
if max_abs > 0:
    y = y / max_abs

plt.figure(figsize=(10, 5))

for N in [256, 512, 1024, 2048, 4096]:

    x = y.copy()

    # If N is larger than signal length, apply zero-padding
    if N > len(x):
        x = np.pad(x, (0, N - len(x)))

    # Apply Hanning window
    x = x * np.hanning(len(x))

    # Compute FFT using current N
    X_fft = np.fft.rfft(x, n=N)
    mag = np.abs(X_fft)
    freqs = np.fft.rfftfreq(N, d=1/sr)

    # Frequency resolution
    delta_f = sr / N
    print(f"N = {N}  →  Δf = {delta_f:.4f} Hz")

    plt.plot(freqs, mag, label=f"N={N} (Δf={delta_f:.1f} Hz)", alpha=0.8)

plt.xlim(0, 800)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("Effect of FFT Size on Frequency Resolution")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plot: Effect of FFT size / zero-padding on spectrum
file_path = DATA_DIR / "0_george_0.wav"

y, sr = librosa.load(file_path, sr=None, mono=True)
y, _ = librosa.effects.trim(y, top_db=20)

max_abs = np.max(np.abs(y))
if max_abs > 0:
    y = y / max_abs

print("Original signal length:", len(y))

plt.figure(figsize=(10, 4))

# Use values larger than signal length to clearly show zero-padding effect
# The following visualization code was created with assistance from ChatGPT
for N in [len(y), 4096, 8192, 16384]:

    x = y.copy()

    if N > len(x):
        x = np.pad(x, (0, N - len(x)))

    x = x * np.hanning(len(x))

    X_fft = np.fft.rfft(x, n=N)
    mag = np.abs(X_fft)
    freqs = np.fft.rfftfreq(N, d=1/sr)

    plt.plot(freqs, mag, label=f"N_FFT={N}", alpha=0.8)

plt.legend()
plt.title("Effect of FFT Size / Zero-Padding on Spectrum (0_george_0.wav)")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.grid(True)
plt.show()

max_mag = 0

results = []

for N in [len(y), 4096, 8192, 16384]:

    x = y.copy()

    if N > len(x):
        x = np.pad(x, (0, N - len(x)))

    x = x * np.hanning(len(x))

    X_fft = np.fft.rfft(x, n=N)
    mag = np.abs(X_fft)
    freqs = np.fft.rfftfreq(N, d=1/sr)

    results.append((freqs, mag, N))

    max_mag = max(max_mag, np.max(mag))

for freqs, mag, N in results:

    plt.figure(figsize=(8,4))
    plt.plot(freqs, mag)

    plt.title(f"Spectrum for N = {N}")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")

    plt.xlim(0, 4000)
    plt.ylim(0, max_mag)

    plt.grid(True)
    plt.show()